# PPO Rolling Notebook (Dow-30, 2021-2023)

This notebook trains and evaluates a PPO agent with rolling windows on Dow-30 stocks.

## Experimental Setting
- Data: daily OHLCV for all 30 Dow Jones stocks
- Date range: 2021-01-01 to 2023-12-01
- Trading days in paper setting: 734
- K = 30 (assets), I = 10 (indicators/features)
- Rolling windows: 252-day train, 20-day validation, 20-day test

Outputs are saved to `./logs/stock/ppo_rolling`.

In [20]:
import os
import multiprocessing as mp
import warnings
import numpy as np
import pandas as pd
import torch
import yfinance as yf
import matplotlib.pyplot as plt

# macOS fork-safety guard for multiprocessing + Objective-C runtime
os.environ.setdefault('OBJC_DISABLE_INITIALIZE_FORK_SAFETY', 'YES')
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

warnings.filterwarnings('ignore')

# Compatibility patch for PyTorch>=2.6 model loading
if not hasattr(torch, '_original_load'):
    torch._original_load = torch.load

def patched_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return torch._original_load(*args, **kwargs)

torch.load = patched_torch_load

# Compatibility patch for yfinance proxy arg
if not hasattr(yf, '_original_download'):
    yf._original_download = yf.download

def patched_download(*args, **kwargs):
    kwargs.pop('proxy', None)
    return yf._original_download(*args, **kwargs)

yf.download = patched_download

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.config_tickers import DOW_30_TICKER
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from elegantrl.agents import AgentPPO
from elegantrl.train.run import train_agent
from env_wrapper.ppo_env_wrapper import ElegantFinRLWrapper

try:
    from elegantrl.train.config import Config as Arguments
except ImportError:
    from elegantrl.train.config import Arguments

print('[OK] Imports complete. spawn start method enabled on macOS.')

[OK] Imports complete. spawn start method enabled on macOS.


In [21]:
# ===== Global config =====
START_DATE = '2020-11-23'
END_DATE = '2023-11-10'

TRAIN_WINDOW = 252
VAL_WINDOW = 20
TEST_WINDOW = 20

K = 30
I = 10

INITIAL_CAPITAL = 1_000_000
HMAX = 100
BUY_COST = 0.001
SELL_COST = 0.001
REWARD_SCALING = 1e-4

# I=10 indicators/features in state representation
ALL_INDICATORS = [
    'macd',
    'boll_ub',
    'boll_lb',
    'rsi_30',
    'cci_30',
    'dx_30',
    'close_30',
    'close_60',
    'volume_30',
    'volume_60',
]

print('[OK] Config set.')

[OK] Config set.


In [22]:
ALL_INDICATORS = ['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30', 'close_60']

def prepare_data(cache_file='./data/stock_dow30_cache.csv',
                 train_start='2021-01-01',
                 test_end='2023-12-01'):
    os.makedirs(os.path.dirname(cache_file), exist_ok=True)

    if os.path.exists(cache_file):
        print(f'[INFO] Loading cache: {cache_file}')
        df = pd.read_csv(cache_file)
        df = df.drop_duplicates(subset=['date', 'tic'])
        df = df[(df['date'] >= START_DATE) & (df['date'] <= END_DATE)].reset_index(drop=True)
        if len(df) > 0:
            print(f'[OK] Cache loaded: {len(df)} rows')
            return df

    print('[INFO] Downloading data from Yahoo Finance...')
    df = YahooDownloader(start_date=train_start, end_date=test_end, ticker_list=DOW_30_TICKER).fetch_data()
    # make sure only the data from start_date to end_date
    df = df[(df['date'] >= train_start) & (df['date'] <= test_end)].reset_index(drop=True)

    tech_indicators = ['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']
    fe = FeatureEngineer(
        use_technical_indicator=True,
        tech_indicator_list=tech_indicators,
        use_vix=False,
        use_turbulence=False,
        user_defined_feature=False,
    )
    df = fe.preprocess_data(df)
    print(df.head())

    if 'close_30_sma' in df.columns:
        df = df.rename(columns={'close_30_sma': 'close_30'})
    if 'close_60_sma' in df.columns:
        df = df.rename(columns={'close_60_sma': 'close_60'})

    df = df.dropna().reset_index(drop=True)
    df.to_csv(cache_file, index=False)
    print(f'[OK] Data cached at: {cache_file}')
    return df

df_raw = prepare_data(train_start=START_DATE, test_end=END_DATE)
df_raw['date'] = pd.to_datetime(df_raw['date'])

# Keep only dates where all tickers exist
df_pivot = df_raw.pivot(index='date', columns='tic', values='close')
valid_dates = df_pivot.dropna().index
df = df_raw[df_raw['date'].isin(valid_dates)].copy()

print(f'[INFO] Filtered rows: {len(df)}')
print(f'[INFO] Date range: {df["date"].min()} to {df["date"].max()}')
print(f'[INFO] Tickers: {df["tic"].nunique()}')

[INFO] Loading cache: ./data/stock_dow30_cache.csv
[OK] Cache loaded: 20916 rows
[INFO] Filtered rows: 20916
[INFO] Date range: 2020-11-23 00:00:00 to 2023-11-10 00:00:00
[INFO] Tickers: 28


In [ ]:
def setup_ppo_args(env_args, cwd_path):
    args = Arguments(agent_class=AgentPPO, env_class=ElegantFinRLWrapper)
    args.env_args = env_args
    args.env_name = env_args['env_name']

    args.net_dims = (128, 64)
    args.state_dim = env_args['state_dim']
    args.action_dim = env_args['action_dim']
    args.if_discrete = env_args['if_discrete']

    args.learning_rate = 1e-4
    args.batch_size = 128
    args.target_step = 2000
    args.break_step = 40000

    args.worker_num = 1
    args.eval_proc_num = 0
    args.if_use_multi_processing = False
    args.eval_gap = 2000
    args.save_gap = 2000
    args.print_verbosity = 1000

    args.if_save = True
    args.if_overwrite_save = True
    args.cwd = cwd_path
    args.if_remove = False
    return args

def run_inference(stock_dim, indicators, args, env_params):
    state_dim = 1 + stock_dim * (len(indicators) + 2)

    env = ElegantFinRLWrapper(**env_params)

    agent = AgentPPO(args.net_dims, args.state_dim, args.action_dim)
    agent.save_or_load_agent(args.cwd, if_save=False)
    agent.act.eval()

    reset_ret = env.reset()
    state = reset_ret[0] if isinstance(reset_ret, tuple) else reset_ret

    done = False
    while not done:
        s_tensor = torch.as_tensor((state,), dtype=torch.float32, device=agent.device)
        with torch.no_grad():
            action = agent.act(s_tensor).detach().cpu().numpy()[0]

        step_ret = env.step(action)
        if len(step_ret) == 5:
            state, reward, term, trunc, _ = step_ret
            done = term or trunc
        else:
            state, reward, done, _, _ = step_ret

    return env.env.save_asset_memory()

print('[OK] PPO helper functions ready.')

[OK] PPO helper functions ready.


In [ ]:
# Rolling window training, validation, testing
# stock_dimension = df['tic'].nunique()
# if stock_dimension != K:
#     print(f'[WARN] K mismatch: expected {K}, got {stock_dimension}')
import tqdm
stock_dimension = 30

unique_dates = sorted(df['date'].unique())
stock_dimension = len(df['tic'].unique())
state_dim = 1 + stock_dimension * (len(ALL_INDICATORS) + 2)



all_test_results = []
last_amount = 1000_000  
cwd_path = f'./checkpoints/ppo_rolling'
os.makedirs(cwd_path, exist_ok=True)

for i in tqdm.tqdm(range(TRAIN_WINDOW + VAL_WINDOW, len(unique_dates) - TEST_WINDOW, TEST_WINDOW), desc='Rolling'):
    train_dates = unique_dates[i - TRAIN_WINDOW - VAL_WINDOW: i - VAL_WINDOW]
    test_dates = unique_dates[i: i + TEST_WINDOW]

    train_df = df[df['date'].isin(train_dates)].sort_values(['date', 'tic']).reset_index(drop=True)
    test_df = df[df['date'].isin(test_dates)].sort_values(['date', 'tic']).reset_index(drop=True)
    

    for d_df in [train_df, test_df]:
        date_map = {d: idx for idx, d in enumerate(sorted(d_df['date'].unique()))}
        d_df['day'] = d_df['date'].map(date_map)
        d_df.set_index('day', inplace=True, drop=False)


    env_params = {
            'env_name': f'FinRL_PPO_Rolling',
            'df': train_df,
            'stock_dim': stock_dimension,
            'hmax': 100,
            'initial_amount': 1000000,
            'num_stock_shares': [0] * stock_dimension,
            'buy_cost_pct': [0.001] * stock_dimension,
            'sell_cost_pct': [0.001] * stock_dimension,
            'reward_scaling': 1e-4,
            'state_space': state_dim,
            'action_space': stock_dimension,
            'tech_indicator_list': ALL_INDICATORS,
            'state_dim': state_dim,
            'action_dim': stock_dimension,
            'if_discrete': False,
            'target_return': 10.0,
        }
    # print(f'\n>>> Rolling window: train {train_dates[0]} to {train_dates[-1]}, test {test_dates[0]} to {test_dates[-1]}')

    try:
        # train
        args = setup_ppo_args(env_params, cwd_path)
        train_agent(args)
        # test
        env_params['df'] = test_df
        env_params['initial_amount'] = float(last_amount)
        result_df = run_inference(stock_dimension, ALL_INDICATORS, args, env_params)

        # Keep next window's initial capital as a scalar, not a Series.
        last_amount = float(result_df['account_value'].iloc[-1])

        result_df = result_df.copy()
        result_df['window_id'] = i
        result_df['window_start'] = pd.to_datetime(test_dates[0])
        result_df['window_end'] = pd.to_datetime(test_dates[-1])
        all_test_results.append(result_df)

    except Exception as e:
        print(f'[ERROR] Window {i} failed: {e}')
        continue

if all_test_results:
    final_results = pd.concat(all_test_results).reset_index(drop=True)
    os.makedirs('./logs/stock', exist_ok=True)
    result_path = './logs/stock/ppo_rolling_results.csv'
    final_results.to_csv(result_path, index=False)
    print(f'[OK] Rolling window results saved: {result_path}')
else:
    print('[WARN] No Rolling window results generated.')

Rolling:   0%|          | 0/23 [00:00<?, ?it/s]

| train_agent_multiprocessing() with GPU_ID 0
| Arguments Keep cwd: ./checkpoints/ppo_rolling
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
| Evaluator:
| `step`: Number of samples, or total training steps, or running times of `env.step()`.
| `time`: Time spent from the start of training to this moment.
| `avgR`: Average value of cumulative rewards, which is the sum of rewards in an episode.
| `stdR`: Standard dev of cumulative rewards, which is the sum of rewards in an episode.
| `avgS`: Average of steps in an episode.
| `objC`: Objective of Critic network. Or call i

Rolling:   4%|▍         | 1/23 [01:52<41:10, 112.29s/it]

| Learner Closed
| Evaluator Closed
| train_agent_multiprocessing() with GPU_ID 0
| Arguments Keep cwd: ./checkpoints/ppo_rolling
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
| Evaluator:
| `step`: Number of samples, or total training steps, or running times of `env.step()`.
| `time`: Time spent from the start of training to this moment.
| `avgR`: Average value of cumulative rewards, which is the sum of rewards in an episode.
| `stdR`: Standard dev of cumulative rewards, which is the sum of rewards in an episode.
| `avgS`: Average of steps in an episode.
| `objC`: Ob

Rolling:   9%|▊         | 2/23 [03:38<38:02, 108.70s/it]

...

| Evaluator Closed
| train_agent_multiprocessing() with GPU_ID 0
| Arguments Keep cwd: ./checkpoints/ppo_rolling
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
| Evaluator:
| `step`: Number of samples, or total training steps, or running times of `env.step()`.
| `time`: Time spent from the start of training to this moment.
| `avgR`: Average value of cumulative rewards, which is the sum of rewards in an episode.
| `stdR`: Standard dev of cumulative rewards, which is the sum of rewards in an episode.
| `avgS`: Average of steps in an episode.
| `objC`: Objective of Critic

Rolling:  96%|█████████▌| 22/23 [44:14<02:10, 130.07s/it]

| Evaluator Closed
| train_agent_multiprocessing() with GPU_ID 0
| Arguments Keep cwd: ./checkpoints/ppo_rolling
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
| Evaluator:
| `step`: Number of samples, or total training steps, or running times of `env.step()`.
| `time`: Time spent from the start of training to this moment.
| `avgR`: Average value of cumulative rewards, which is the sum of rewards in an episode.
| `stdR`: Standard dev of cumulative rewards, which is the sum of rewards in an episode.
| `avgS`: Average of steps in an episode.
| `objC`: Objective of Critic

Rolling: 100%|██████████| 23/23 [46:30<00:00, 121.31s/it]

| Learner Closed
[OK] Rolling window results saved: ./logs/stock/ppo_rolling_results.csv


In [25]:
def calculate_metrics(series):
    series = pd.Series(series).dropna().astype(float)
    if len(series) < 2:
        return 0.0, 0.0, 0.0

    daily_ret = series.pct_change().dropna()
    cum_ret = (series.iloc[-1] / series.iloc[0]) - 1
    sharpe = (252 ** 0.5) * daily_ret.mean() / daily_ret.std() if daily_ret.std() != 0 else 0.0
    rolling_max = series.cummax()
    drawdown = (series - rolling_max) / rolling_max
    mdd = drawdown.min()
    return float(cum_ret), float(sharpe), float(mdd)

def stitch_window_curve(result_df, value_col='account_value', initial_capital=None):
    """Compound per-window curves into one continuous equity curve for fair benchmark comparison."""
    df_local = result_df.copy().sort_values('date').reset_index(drop=True)
    if initial_capital is None:
        initial_capital = float(df_local[value_col].iloc[0])

    if 'window_id' not in df_local.columns:
        return pd.Series(df_local[value_col].values, index=df_local.index, dtype=float)

    stitched_parts = []
    capital = float(initial_capital)
    for _, wdf in df_local.groupby('window_id', sort=True):
        wdf = wdf.sort_values('date')
        base = float(wdf[value_col].iloc[0])
        if base == 0:
            scaled = pd.Series([capital] * len(wdf), index=wdf.index, dtype=float)
        else:
            scaled = capital * (wdf[value_col].astype(float) / base)
        capital = float(scaled.iloc[-1])
        stitched_parts.append(scaled)

    stitched = pd.concat(stitched_parts).sort_index()
    return stitched

def evaluate_result(result_file='./logs/stock/ppo_rolling_results.csv'):
    if not os.path.exists(result_file):
        print(f'[WARN] Result file not found: {result_file}')
        return

    result_df = pd.read_csv(result_file)
    required_cols = {'date', 'account_value'}
    if not required_cols.issubset(result_df.columns):
        print(f'[WARN] Missing required columns: {required_cols - set(result_df.columns)}')
        return

    result_df['date'] = pd.to_datetime(result_df['date'])
    result_df = result_df.sort_values('date').reset_index(drop=True)

    initial_capital = float(result_df['account_value'].iloc[0])
    result_df['stitched_account_value'] = stitch_window_curve(
        result_df,
        value_col='account_value',
        initial_capital=initial_capital,
    )

    start_date = result_df['date'].iloc[0].strftime('%Y-%m-%d')
    end_date_inclusive = result_df['date'].iloc[-1]
    benchmark_end = (end_date_inclusive + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    benchmark = yf.download('^DJI', start=start_date, end=benchmark_end, progress=False)
    if isinstance(benchmark.columns, pd.MultiIndex):
        benchmark.columns = benchmark.columns.get_level_values(0)
    benchmark = benchmark.reset_index()
    if 'Date' not in benchmark.columns and 'index' in benchmark.columns:
        benchmark = benchmark.rename(columns={'index': 'Date'})
    benchmark['Date'] = pd.to_datetime(benchmark['Date'])

    merged = pd.merge(result_df, benchmark[['Date', 'Close']], left_on='date', right_on='Date', how='left')
    merged['Close'] = merged['Close'].ffill().bfill()

    if merged['Close'].isna().all():
        print('[WARN] Benchmark close prices are empty after alignment.')
        return

    merged = merged.dropna(subset=['Close']).reset_index(drop=True)
    merged['benchmark_value'] = (merged['Close'] / merged['Close'].iloc[0]) * initial_capital

    ppo_ret, ppo_sharpe, ppo_mdd = calculate_metrics(merged['stitched_account_value'])
    bm_ret, bm_sharpe, bm_mdd = calculate_metrics(merged['benchmark_value'])

    print('\n' + '=' * 74)
    print('PPO 252/20/20 Rolling Backtest Report (Window-Compounded)')
    print('=' * 74)
    print(f'Cumulative Return  | PPO: {ppo_ret * 100:8.2f}% | Benchmark: {bm_ret * 100:8.2f}%')
    print(f'Sharpe Ratio       | PPO: {ppo_sharpe:8.3f} | Benchmark: {bm_sharpe:8.3f}')
    print(f'Max Drawdown       | PPO: {ppo_mdd * 100:8.2f}% | Benchmark: {bm_mdd * 100:8.2f}%')
    print('=' * 74)

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(merged['date'], merged['stitched_account_value'], label='PPO Portfolio (Compounded)', color='#2E86AB', linewidth=2.2)
    ax.plot(merged['date'], merged['benchmark_value'], label='Dow Jones Benchmark', color='#A23B72', linewidth=1.7, linestyle='--')
    ax.set_title('PPO Rolling Window Performance vs Dow Jones')
    ax.set_xlabel('Date')
    ax.set_ylabel('Portfolio Value')
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig_path = './logs/stock/ppo_rolling_backtest.png'
    plt.tight_layout()
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f'[OK] Plot saved: {fig_path}')
    plt.show()

evaluate_result()


PPO 252/20/20 Rolling Backtest Report (Window-Compounded)
Cumulative Return  | PPO:   -11.27% | Benchmark:    -7.35%
Sharpe Ratio       | PPO:   -0.315 | Benchmark:   -0.170
Max Drawdown       | PPO:   -21.53% | Benchmark:   -21.94%
[OK] Plot saved: ./logs/stock/ppo_rolling_backtest.png
